In [0]:
-- Ensure Gold schema exists
CREATE SCHEMA IF NOT EXISTS weather_openmeteo.gold;

USE weather_openmeteo.gold;

-- Create Gold KPIs table with explicit schema and Delta format
CREATE TABLE IF NOT EXISTS weather_openmeteo.gold.weather_daily_kpis (
    date TIMESTAMP,
    latitude DOUBLE,
    longitude DOUBLE,
    city STRING,
    temperature_2m_max DOUBLE,
    temperature_2m_min DOUBLE,
    daylight_duration DOUBLE,
    wind_speed_10m_max DOUBLE,
    avg_temperature_7d DOUBLE,
    alerts_temp BOOLEAN,
    alerts_wind BOOLEAN
)
USING DELTA;

-- Prepare Gold KPIs with windowed metrics and alert logic
WITH gold_kpis AS (
    SELECT 
        date,
        latitude,
        longitude,
        city,
        temperature_2m_max,
        temperature_2m_min,
        daylight_duration,
        wind_speed_10m_max,
        -- 7-day forward-looking average temperature
        AVG(temperature_2m_max) OVER (
            PARTITION BY city, latitude, longitude
            ORDER BY CAST(date AS DATE)
            ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING
        ) AS avg_temperature_7d,
        -- Temperature alert: true if max temp > 103°F or < -20°F
        CASE 
            WHEN temperature_2m_max < -20.0 OR temperature_2m_max > 103.0 THEN true 
            ELSE false 
        END AS alerts_temp,
        -- Wind alert: true if max wind speed > 40 mph
        CASE 
            WHEN wind_speed_10m_max > 40.0 THEN true 
            ELSE false 
        END AS alerts_wind
    FROM weather_openmeteo.silver.weather_daily_clean
)

-- Upsert Gold KPIs table with new and updated metrics
MERGE INTO weather_openmeteo.gold.weather_daily_kpis AS target
USING gold_kpis AS source
ON target.date = source.date 
   AND target.latitude = source.latitude 
   AND target.longitude = source.longitude

WHEN MATCHED THEN
  UPDATE SET
    target.city = source.city,
    target.temperature_2m_max = source.temperature_2m_max,
    target.temperature_2m_min = source.temperature_2m_min,
    target.daylight_duration = source.daylight_duration,
    target.wind_speed_10m_max = source.wind_speed_10m_max,
    target.avg_temperature_7d = CAST(source.avg_temperature_7d AS DOUBLE),
    target.alerts_temp = source.alerts_temp,
    target.alerts_wind = source.alerts_wind

WHEN NOT MATCHED THEN
  INSERT (
    date, latitude, longitude, city, temperature_2m_max, temperature_2m_min, 
    daylight_duration, wind_speed_10m_max, avg_temperature_7d, alerts_temp, alerts_wind
  )
  VALUES (
    source.date, source.latitude, source.longitude, source.city, source.temperature_2m_max, 
    source.temperature_2m_min, source.daylight_duration, source.wind_speed_10m_max, 
    CAST(source.avg_temperature_7d AS DOUBLE), source.alerts_temp, source.alerts_wind
  );

-- Display Gold KPIs table for validation
SELECT * FROM weather_openmeteo.gold.weather_daily_kpis;

